# TICKET MJD-604
De: Carmen Ruiz · Dirección general
Vamos a negociar con Old Pier, el proveedor escocés del whisky, que factura en libras. Necesito la me‐
dia mensual del cambio euro-libra de 2025, y cuánto nos habría costado un pedido fijo de 10.000 £ cada
mes al cambio medio de ese mes.

ENTREGABLE
- entregables/mjd-604_cambio_gbp_2025.csv , con el mes, el cambio medio y el coste en euros.

In [ ]:
import os, sys
from pathlib import Path

RAIZ = Path.home() / "master-data-ia"
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

In [2]:
import time
import requests
import pandas as pd
import duckdb

pd.reset_option("display.float_format")
carpeta = Path("datos/exportaciones")

def tipos_de_cambio(anio, moneda="GBP"):
    r = requests.get(
        f"https://api.frankfurter.dev/v1/{anio}-01-01..{anio}-12-31",
        params={"base": "EUR", "symbols": moneda},
        timeout=10,
    )
    r.raise_for_status()
    serie = pd.Series({f: t[moneda] for f, t in r.json()["rates"].items()}, name=moneda)
    serie.index = pd.to_datetime(serie.index)
    return serie[serie.index.year == anio]

def tipos_con_cache(anio, moneda="GBP"):
    fichero = carpeta / f"tipos_{moneda}_{anio}.csv"
    if fichero.exists():
        return pd.read_csv(fichero, index_col=0, parse_dates=True).squeeze("columns")
    serie = tipos_de_cambio(anio, moneda)
    serie.to_csv(fichero)
    return serie

gbp = tipos_con_cache(2025)
print(len(gbp), gbp.index.min().date(), gbp.index.max().date())

tabla = gbp.resample("MS").agg(["mean", "count"])
tabla.columns = ["gbp_por_eur", "dias"]
tabla["coste_eur"] = 10_000 / tabla["gbp_por_eur"]
tabla.round({"gbp_por_eur": 5, "coste_eur": 2})

255 2025-01-02 2025-12-31


,gbp_por_eur,dias,coste_eur
2025-01-01,0.83908,22,11917.80
2025-02-01,0.83071,20,12037.90
2025-03-01,0.83703,21,11947.07
2025-04-01,0.85379,20,11712.53
2025-05-01,0.84350,21,11855.43
2025-06-01,0.84981,21,11767.34
2025-07-01,0.86469,23,11564.88
2025-08-01,0.86528,21,11557.00
2025-09-01,0.86895,22,11508.20
2025-10-01,0.87155,23,11473.78


El tipo es «libras por euro» (un euro compra 0,8x libras), así que 10.000 £ cuestan 10000 / tipo euros: cuan‐
to más baja el tipo, más caro sale el whisky. Y el resumen para Carmen, más la segunda vía: la media anual
de los días debe estar cerca de la media de las medias mensuales (no es idéntica, porque cada mes tiene dis‐
tinto número de días):

In [4]:
coste = tabla["coste_eur"]
print(f"Más barato: {coste.idxmin():%B} ({coste.min():,.2f} €)")
print(f"Más caro: {coste.idxmax():%B} ({coste.max():,.2f} €)")
print(f"Total 2025: {coste.sum():,.2f} € por 120.000 £")
print(round(gbp.mean(), 5), round(tabla["gbp_por_eur"].mean(), 5))
tabla.reset_index(names="mes").to_csv("entregables/mjd-604_cambio_gbp_2025.csv", index=False)

Más barato: November (11,364.09 €)
Más caro: February (12,037.90 €)
Total 2025: 140,134.60 € por 120.000 £
0.85679 0.85661
